### ML Lab: example of optuna's hyper-parameter search using random forests

The script optimizes hyperparameters for random forest model based on a training partition using OOB score as validation score.
The final selected RF is evaluated on the independent test partition.

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.8 MB/s eta 0:00:00


In [3]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare independent sets
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


def objective(trial):
    # 2. Define search space
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 32, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "bootstrap": True,
        "oob_score": True,
        "n_jobs": -1,
        "random_state": 42
    }

    # 3. Fit on training subset only
    clf = RandomForestClassifier(**params)
    clf.fit(X_train, y_train)

    # 4. Return OOB score (Selection Metric)
    return clf.oob_score_

# 5. Execute Hyperparameter Search
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# 6. Best model found
print(f"Best OOB Score: {study.best_value:.4f}")
print(f"Best Params: {study.best_params}")

[I 2026-04-24 15:53:50,089] A new study created in memory with name: no-name-e0367037-d55b-4efa-ac36-2bd059f2b68b
[I 2026-04-24 15:53:50,584] Trial 0 finished with value: 0.9538461538461539 and parameters: {'n_estimators': 151, 'max_depth': 27, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.9538461538461539.
[I 2026-04-24 15:53:52,005] Trial 1 finished with value: 0.9626373626373627 and parameters: {'n_estimators': 452, 'max_depth': 18, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.9626373626373627.
[I 2026-04-24 15:53:53,232] Trial 2 finished with value: 0.9406593406593406 and parameters: {'n_estimators': 440, 'max_depth': 2, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.9626373626373627.
[I 2026-04-24 15:53:53,673] Trial 3 finished with value: 0.9384615384615385 and parameters: {'n_estimators': 157, 'max_depth': 2, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.9626373626373627.
[I 2026-04-24 15:53:54,596] Trial 4 finished with value: 0.9582417582417583 

Best OOB Score: 0.9692
Best Params: {'n_estimators': 324, 'max_depth': 11, 'min_samples_leaf': 3}


#### Some useful visualizations of the search

In [4]:
import optuna.visualization as vis

# Visualize the trade-offs found using OOB validation
vis.plot_parallel_coordinate(study).show()
vis.plot_optimization_history(study).show()
vis.plot_contour(study, params=["max_depth", "n_estimators"]).show()

#### Final model assessment

We use the test partition to assess the quality of the final selected model

In [5]:
# Re-train or simply use the best parameters to predict on the test set
best_clf = RandomForestClassifier(**study.best_params, n_jobs=-1)
best_clf.fit(X_train, y_train)

test_preds = best_clf.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print(f"Final Independent Test Accuracy: {test_acc:.4f}")
print("\nFinal Classification Report:\n", classification_report(y_test, test_preds))

Final Independent Test Accuracy: 0.9649

Final Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



#### Questions:

- Is using the test-set accuracy as criterion to optimize hyper-parameter search a good idea? Why?

- Is using OOB score over the training set a good idea as criterion to optimize hyper-parameter search? Why?

- Is using the training-set accuracy as criterion to optimize hyper-parameter search a good idea? Why?

## Proposed exercise:

Optimize a variety of models you have seen so far for classification (logistic regression, regularized, LDA/QDA, k-NN, and random forests) using optuna to select best hyper-parameters, then select your final best model based on cross-validation or similar variant, and finally assess your final model's performance on an independent test set.